In [1]:
import json
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from datasets import Dataset
from torch.utils.data import DataLoader
import torch
import pyro
import pyro.distributions as dist
from tqdm import tqdm
import os

file_path = "/home/panshul2/cs-546/Iterative-SSR-and-EVCL-Catastrophic-Forgetting/SSR/Latest_Weights/QA_QG_SA_Weights/task1312_amazonreview_polarity_classification.json"
with open(file_path, "r") as f:
    data = json.load(f)

instruct1=(
    "\nInstruction: You will be given a sentence describing a review. "
    "Classify its sentiment as either 'positive' or 'negative'. "
    "Respond only with the sentiment label ('positive' or 'negative') without any additional information. "
    "Ensure the output is concise, accurate, and adheres to the classification labels. "
    "\nReview: "
)
instruct2="\nSentiment: "
instances = data["Instances"][4500:5000]
test_inputs = [instruct1+instance["input"]+instruct2 for instance in instances]
test_outputs = [instance["output"][0] for instance in instances]


base_model_path = "meta-llama/Meta-Llama-3-8B"  
tokenizer = AutoTokenizer.from_pretrained(base_model_path)


if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def tokenize_function(examples):
    model_inputs = tokenizer(
        examples["input"],
        truncation=True,
        padding="max_length",
        max_length=512,
    )
    labels = tokenizer(
        examples["output"],
        truncation=True,
        padding="max_length",
        max_length=512,
    )["input_ids"]
    model_inputs["labels"] = labels
    return model_inputs

batch_size = 16  

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True,
)
fine_tuned_weights_path = "/home/panshul2/cs-546/Iterative-SSR-and-EVCL-Catastrophic-Forgetting/SSR/Synthethic_Data_Generation/fine_tuned_sa_noEVCL2/fine-tuned-sa-lora_noEVCL2"

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    quantization_config=bnb_config,
    device_map="auto"
)
model = PeftModel.from_pretrained(base_model, fine_tuned_weights_path)

DEVICE = model.device

# Generate predictions
predictions = []
references = []

output_file_path = "/home/panshul2/cs-546/Iterative-SSR-and-EVCL-Catastrophic-Forgetting/predictions_LORA_Task2_Final_SA2.json"

if os.path.exists(output_file_path):
    with open(output_file_path, "r") as f:
        saved_data = json.load(f)
else:
    saved_data = {"predictions": [], "references": []}

print("Generating predictions incrementally:")

print("Generating predictions:")

for i in tqdm(range(0, len(test_inputs), batch_size)):  # Loop in batches
    batch_inputs = test_inputs[i:i + batch_size]
    batch_references = test_outputs[i:i + batch_size]

    # Tokenize the inputs in a batch
    inputs_tokenized = tokenizer(batch_inputs, padding=True, truncation=True, return_tensors="pt").to(DEVICE)

    with torch.no_grad():

        # Generate predictions using the tokenized inputs
        generated_ids = model.generate(
                input_ids=inputs_tokenized["input_ids"],
                attention_mask=inputs_tokenized["attention_mask"],
                max_new_tokens=30,  
                min_length=10,  
                no_repeat_ngram_size=2,  
                num_return_sequences=1,
                top_p=0.9,  
                temperature=0.7  
            )
        
    batch_predictions = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
    predictions.extend(batch_predictions)
    references.extend(batch_references)

    saved_data["predictions"].extend(batch_predictions)
    saved_data["references"].extend(batch_references)

    # Save incrementally to JSON
    with open(output_file_path, "w") as json_file:
        json.dump(saved_data, json_file, indent=4)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Generating predictions incrementally:
Generating predictions:


  0%|          | 0/32 [00:00<?, ?it/s]Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
  3%|▎         | 1/32 [00:39<20:32, 39.76s/it]Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
  6%|▋         | 2/32 [01:17<19:15, 38.52s/it]Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenize

In [2]:
import evaluate

rouge = evaluate.load("rouge")
results = rouge.compute(predictions=predictions, references=references)

# Display the results
print("\nROUGE Scores:")
print(results)


ROUGE Scores:
{'rouge1': np.float64(0.01668067390829497), 'rouge2': np.float64(0.0), 'rougeL': np.float64(0.016690202073691995), 'rougeLsum': np.float64(0.016681094016521153)}


In [2]:
!pip install evaluate

In [ ]:
import json
import evaluate

# Path to the saved predictions file
output_file_path = "/home/kag12/Iterative-SSR-and-EVCL-Catastrophic-Forgetting/predictions_LORA_Task2_Final_QA.json"

# Load the saved predictions and references
with open(output_file_path, "r") as f:
    saved_data = json.load(f)

predictions = saved_data["predictions"]
references = saved_data["references"]

# Ensure predictions and references are aligned in length
assert len(predictions) == len(references), "Predictions and references length mismatch!"

# Load the ROUGE evaluation metric
rouge = evaluate.load("rouge")

# Compute ROUGE scores
results = rouge.compute(predictions=predictions, references=references)

# Display the ROUGE scores
print("\nROUGE Scores:")
print(results)
